# DiffuSVG — Dataset Builder

Builds a better `training_pairs.json` for the v7 pipeline by:
1. Reading `results.json` from a previous run
2. Mining prompts where CLIP < 24 or DINO < 0.35 or success=False
3. Generating better SVGs for those prompts via SD 3.5 Medium → Potrace
4. Merging with the existing seed `training_pairs.json`
5. Downloading the final `training_pairs.json` ready for the v7 Colab

---
⚠️ **Before you start:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run **Cell 1 (Install)** then `Runtime → Restart runtime`
3. Skip Cell 1 and run the rest

In [ ]:
#@title Step 1 — Install  (run once, then restart runtime)
import subprocess, sys

# System: potrace + imagemagick
subprocess.check_call(["apt-get", "install", "-qq", "potrace", "imagemagick"])

# Python
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "diffusers>=0.27",
    "transformers>=4.37",
    "accelerate>=0.26",
    "torch",
    "pillow",
    "numpy",
    "tqdm",
])

print("\n✅ Done. Now go to  Runtime → Restart runtime  then continue from Step 2.")

In [ ]:
#@title Step 2 — Verify GPU + Tools
import subprocess, shutil, torch

# GPU
if not torch.cuda.is_available():
    raise RuntimeError("No GPU — go to Runtime → Change runtime type → T4 GPU")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Potrace
if not shutil.which("potrace"):
    raise EnvironmentError("potrace not found — re-run Step 1")
v = subprocess.check_output(["potrace", "--version"]).decode().split("\n")[0]
print(f"potrace: {v}")

# ImageMagick (optional but preferred)
if shutil.which("convert"):
    v2 = subprocess.check_output(["convert", "--version"]).decode().split("\n")[0]
    print(f"ImageMagick: {v2}")
else:
    print("ImageMagick: not found (will use PIL fallback — still works)")

print("\n✅ All tools present.")

In [ ]:
#@title Step 3 — Upload Files
#
# Required : results.json          (from your previous DiffuSVG run)
# Optional : training_pairs.json   (existing seed dataset — will be merged)

import os
from google.colab import files

print("Select files to upload (hold Ctrl/Cmd for multiple):")
uploaded = files.upload()

RESULTS_PATH         = None
EXISTING_PAIRS_PATH  = None

for fname, data in uploaded.items():
    dest = f"/content/{fname}"
    with open(dest, "wb") as f:
        f.write(data)
    print(f"  Saved: {dest}")
    if "results" in fname and fname.endswith(".json"):
        RESULTS_PATH = dest
    if "training_pairs" in fname and fname.endswith(".json"):
        EXISTING_PAIRS_PATH = dest

if RESULTS_PATH is None:
    raise ValueError("results.json not found in uploaded files. Upload it and re-run.")

print(f"\nresults.json         : {RESULTS_PATH}")
print(f"training_pairs.json  : {EXISTING_PAIRS_PATH or 'not uploaded (ok — will create fresh)'}")

In [ ]:
#@title Step 4 — Configuration  { display-mode: "form" }

# Failure thresholds — prompts scoring below these get regenerated
CLIP_THRESHOLD = 24.0  #@param {type:"number"}
DINO_THRESHOLD = 0.35  #@param {type:"number"}

# HuggingFace token (needed to download SD 3.5 Medium)
# Get yours at https://huggingface.co/settings/tokens
HF_TOKEN = ""  #@param {type:"string"}

# SD 3.5 generation settings
DIFFUSION_STEPS    = 30  #@param {type:"slider", min:10, max:50, step:5}
DIFFUSION_GUIDANCE = 5.0  #@param {type:"number"}

# Vectorizer settings
RESOLUTION   = 512  #@param {type:"slider", min:256, max:1024, step:128}
THRESHOLD    = 0.5  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
USE_COLOUR   = False  #@param {type:"boolean"}
NUM_COLOURS  = 6     #@param {type:"slider", min:2, max:16, step:1}

# Output quality filter (structural only — no Qwen2-VL to save VRAM)
MIN_PATH_ELEMENTS = 1   #@param {type:"slider", min:1, max:10, step:1}
MAX_PATH_ELEMENTS = 500 #@param {type:"slider", min:50, max:2000, step:50}

SEED = 42  #@param {type:"number"}

print("✅ Config saved.")

In [ ]:
#@title Step 5 — Mine Bad Prompts from results.json
import json, csv
from pathlib import Path

path = Path(RESULTS_PATH)
with open(path, encoding="utf-8") as f:
    data = json.load(f)

# Support both {"results": [...]} and plain list
records = data["results"] if isinstance(data, dict) and "results" in data else data

bad_prompts = []
breakdown   = {"failed": 0, "low_clip": 0, "low_dino": 0, "total": len(records)}

for r in records:
    bad = False
    if not r.get("success", True):
        breakdown["failed"] += 1
        bad = True
    if float(r.get("clip", 0)) < CLIP_THRESHOLD:
        if r.get("success", True):
            breakdown["low_clip"] += 1
        bad = True
    if float(r.get("dino", 0)) < DINO_THRESHOLD:
        if r.get("success", True) and float(r.get("clip", 0)) >= CLIP_THRESHOLD:
            breakdown["low_dino"] += 1
        bad = True
    if bad:
        bad_prompts.append(r["prompt"])

breakdown["bad_total"] = len(bad_prompts)

print(f"Total prompts in results.json : {breakdown['total']}")
print(f"Outright failures             : {breakdown['failed']}")
print(f"Low CLIP (<{CLIP_THRESHOLD})           : {breakdown['low_clip']}")
print(f"Low DINO (<{DINO_THRESHOLD})           : {breakdown['low_dino']}")
print(f"\nPrompts to regenerate         : {len(bad_prompts)}")
print("\nBad prompts:")
for p in bad_prompts:
    print(f"  • {p}")

if not bad_prompts:
    print("\nNo bad prompts found — thresholds may be too lenient.")
    print("Lower CLIP_THRESHOLD or raise DINO_THRESHOLD in Step 4 to mine more.")

In [ ]:
#@title Step 6 — Load Vectorizer Code (Potrace wrapper)
import io, os, re, shutil, subprocess, tempfile, logging
import numpy as np
from pathlib import Path
from typing import Optional
from PIL import Image

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("Dataset")


class Vectorizer:
    """PIL Image → SVG via Potrace + optional ImageMagick preprocessing."""

    def __init__(self, threshold=0.5, turdsize=2, alphamax=1.0,
                 opttolerance=0.2, resolution=512, invert=False,
                 contrast_stretch="2%x98%"):
        if not shutil.which("potrace"):
            raise EnvironmentError("potrace not found — re-run Step 1")
        self._has_imagemagick = bool(shutil.which("convert"))
        self.threshold        = threshold
        self.turdsize         = turdsize
        self.alphamax         = alphamax
        self.opttolerance     = opttolerance
        self.resolution       = resolution
        self.invert           = invert
        self.contrast_stretch = contrast_stretch

    def vectorize(self, image: Image.Image, prompt: str = "") -> Optional[str]:
        with tempfile.TemporaryDirectory() as tmpdir:
            bmp_path = os.path.join(tmpdir, "input.bmp")
            svg_path = os.path.join(tmpdir, "output.svg")
            self._to_bmp(image, bmp_path)
            if not self._run_potrace(bmp_path, svg_path):
                return None
            with open(svg_path, "r", encoding="utf-8") as f:
                svg = f.read()
            return self._postprocess(svg)

    def _to_bmp(self, image, path):
        if self._has_imagemagick:
            self._to_bmp_imagemagick(image, path)
        else:
            self._to_bmp_pil(image, path)

    def _to_bmp_imagemagick(self, image, path):
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
            png_path = tmp.name
        image.convert("RGB").save(png_path)
        threshold_pct = f"{int(self.threshold * 100)}%"
        cmd = ["convert", png_path,
               "-resize", f"{self.resolution}x{self.resolution}!",
               "-colorspace", "Gray"]
        if self.contrast_stretch:
            cmd += ["-level", self.contrast_stretch]
        if self.invert:
            cmd += ["-negate"]
        cmd += ["-threshold", threshold_pct, "-type", "Bilevel", f"BMP3:{path}"]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode != 0:
            self._to_bmp_pil(image, path)
        os.unlink(png_path)

    def _to_bmp_pil(self, image, path):
        img  = image.convert("RGB").resize((self.resolution, self.resolution), Image.LANCZOS)
        grey = np.array(img.convert("L")).astype(np.float32) / 255.0
        binary = (grey < self.threshold).astype(np.uint8)
        if self.invert:
            binary = 1 - binary
        Image.fromarray((binary * 255).astype(np.uint8), mode="L").convert("1").save(path, format="BMP")

    def _run_potrace(self, bmp_path, svg_path) -> bool:
        cmd = ["potrace", bmp_path, "--svg",
               f"--turdsize={self.turdsize}",
               f"--alphamax={self.alphamax}",
               f"--opttolerance={self.opttolerance}",
               "--output", svg_path]
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
            return r.returncode == 0
        except Exception as e:
            log.warning(f"Potrace error: {e}")
            return False

    def _postprocess(self, svg: str) -> str:
        svg = re.sub(r'<\?xml[^>]*\?>', '', svg)
        svg = re.sub(r'<!DOCTYPE[^>]*>', '', svg)
        svg = re.sub(r'<!--.*?-->', '', svg, flags=re.DOTALL)
        svg = re.sub(r'<metadata\b[^>]*>.*?</metadata>', '', svg, flags=re.DOTALL)
        vb  = re.search(r'viewBox="([^"]+)"', svg)
        if vb:
            parts = vb.group(1).split()
            orig_w = float(parts[2]) if len(parts) == 4 else self.resolution
            orig_h = float(parts[3]) if len(parts) == 4 else self.resolution
        else:
            orig_w = orig_h = float(self.resolution)
        scale_x = 200.0 / orig_w if orig_w > 0 else 1.0
        scale_y = 200.0 / orig_h if orig_h > 0 else 1.0
        svg = re.sub(r'<svg[^>]*>',
                     '<svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">',
                     svg, count=1)
        if abs(scale_x - 1.0) > 0.01 or abs(scale_y - 1.0) > 0.01:
            svg = svg.replace(
                '<svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">',
                f'<svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">'
                f'<g transform="scale({scale_x:.4f},{scale_y:.4f})">',
            )
            svg = svg.replace('</svg>', '</g></svg>')
        return "\n".join(l for l in svg.splitlines() if l.strip()).strip()


class ColourVectorizer(Vectorizer):
    """Multi-pass colour vectorizer using ImageMagick colour quantisation."""

    def __init__(self, num_colors=8, **kwargs):
        super().__init__(**kwargs)
        if not self._has_imagemagick:
            raise EnvironmentError("ImageMagick required for ColourVectorizer")
        self.num_colors = num_colors

    def vectorize(self, image: Image.Image, prompt: str = "") -> Optional[str]:
        with tempfile.TemporaryDirectory() as tmpdir:
            input_path  = os.path.join(tmpdir, "input.png")
            quant_path  = os.path.join(tmpdir, "quantised.png")
            image.convert("RGB").resize((self.resolution, self.resolution), Image.LANCZOS).save(input_path)
            subprocess.run(["convert", input_path, "+dither",
                            "-colors", str(self.num_colors), quant_path],
                           capture_output=True, timeout=30)
            if not os.path.exists(quant_path):
                return super().vectorize(image, prompt)

            result = subprocess.run(["convert", quant_path, "-format", "%c", "histogram:info:-"],
                                    capture_output=True, text=True, timeout=15)
            colors = self._parse_palette(result.stdout)
            if not colors:
                return super().vectorize(image, prompt)

            svg_layers = []
            quant_arr  = np.array(Image.open(quant_path).convert("RGB"))
            for hex_color, (r, g, b) in colors:
                tol  = 20
                mask = ((np.abs(quant_arr[:,:,0].astype(int)-r)<tol) &
                        (np.abs(quant_arr[:,:,1].astype(int)-g)<tol) &
                        (np.abs(quant_arr[:,:,2].astype(int)-b)<tol))
                if mask.sum() < 50: continue
                bmp = os.path.join(tmpdir, f"l_{hex_color[1:]}.bmp")
                svp = os.path.join(tmpdir, f"l_{hex_color[1:]}.svg")
                Image.fromarray((mask*255).astype(np.uint8)).convert("1").save(bmp, format="BMP")
                if not self._run_potrace(bmp, svp): continue
                with open(svp) as f: layer = f.read()
                paths = re.findall(r'<path[^>]*/>', layer, re.DOTALL)
                for p in paths:
                    p = re.sub(r'fill="[^"]*"', '', p)
                    svg_layers.append(p.replace('<path', f'<path fill="{hex_color}"', 1))

            if not svg_layers:
                return super().vectorize(image, prompt)
            return ('<svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">\n'
                    + "\n".join(svg_layers) + "\n</svg>")

    def _parse_palette(self, text):
        seen, out = set(), []
        for line in text.splitlines():
            m = re.search(r'#([0-9A-Fa-f]{6})', line)
            if m:
                h = "#" + m.group(1).upper()
                if h not in seen:
                    seen.add(h)
                    out.append((h, (int(m.group(1)[0:2],16),
                                   int(m.group(1)[2:4],16),
                                   int(m.group(1)[4:6],16))))
        return out


def is_good_svg(svg: str, min_elements=1, max_elements=500) -> bool:
    if not svg or "<svg" not in svg: return False
    paths = len(re.findall(r'<path', svg))
    if paths < min_elements or paths > max_elements: return False
    d_matches = re.findall(r'd="([^"]+)"', svg)
    return bool(d_matches) and max(len(d) for d in d_matches) >= 10


print("✅ Vectorizer loaded.")

In [ ]:
#@title Step 7 — Load SD 3.5 Medium  (~3-5 min, ~8 GB VRAM)
import gc, torch
from diffusers import StableDiffusion3Pipeline

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is empty. Set it in Step 4.\n"
        "Get a free token at https://huggingface.co/settings/tokens\n"
        "You also need to accept the SD3.5 license at:\n"
        "  https://huggingface.co/stabilityai/stable-diffusion-3.5-medium"
    )

SVG_STYLE_PREFIX = (
    "ultra-simple flat vector, geometric shapes only, "
    "solid colors, icon style, minimalist, "
)
NEG_PROMPT = (
    "gradient, shadow, texture, 3d, realistic, photograph, "
    "complex details, noise, grain, watermark"
)

print("Loading SD 3.5 Medium (takes ~3-5 min)...")
pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
pipe.enable_model_cpu_offload()
print("✅ SD 3.5 Medium loaded (CPU-offloaded).")


@torch.inference_mode()
def generate_image(prompt: str, seed: int = None) -> "Image.Image":
    gen = torch.Generator("cuda").manual_seed(seed) if seed is not None else None
    return pipe(
        SVG_STYLE_PREFIX + prompt,
        negative_prompt=NEG_PROMPT,
        num_inference_steps=DIFFUSION_STEPS,
        guidance_scale=DIFFUSION_GUIDANCE,
        generator=gen,
    ).images[0]

In [ ]:
#@title Step 8 — Generate Corrective SVGs
import json, gc, re
from tqdm.auto import tqdm

if not bad_prompts:
    print("No bad prompts to process. Lower the thresholds in Step 4.")
else:
    vec = ColourVectorizer(
        num_colors=NUM_COLOURS, resolution=RESOLUTION, threshold=THRESHOLD
    ) if USE_COLOUR else Vectorizer(resolution=RESOLUTION, threshold=THRESHOLD)

    new_pairs = []
    failed    = []
    stats     = {"ok": 0, "fail_diffusion": 0, "fail_vectorize": 0, "fail_quality": 0}

    for i, prompt in enumerate(tqdm(bad_prompts, desc="Generating")):
        seed = SEED + i

        # Diffusion
        try:
            img = generate_image(prompt, seed=seed)
        except Exception as e:
            log.warning(f"[{i}] Diffusion failed '{prompt}': {e}")
            stats["fail_diffusion"] += 1
            failed.append({"prompt": prompt, "reason": "diffusion"})
            continue

        # Vectorize
        svg = vec.vectorize(img, prompt=prompt)
        if svg is None:
            log.warning(f"[{i}] Vectorization failed '{prompt}'")
            stats["fail_vectorize"] += 1
            failed.append({"prompt": prompt, "reason": "vectorize"})
            continue

        # Structural quality check
        if not is_good_svg(svg, min_elements=MIN_PATH_ELEMENTS, max_elements=MAX_PATH_ELEMENTS):
            log.info(f"[{i}] Quality check failed '{prompt}'")
            stats["fail_quality"] += 1
            failed.append({"prompt": prompt, "reason": "quality"})
            continue

        n_paths = len(re.findall(r'<path', svg))
        new_pairs.append({"prompt": prompt, "svg": svg, "is_seed": False,
                          "source": "sd35+potrace", "svg_chars": len(svg)})
        stats["ok"] += 1
        log.info(f"[{i}] OK  '{prompt}'  ({len(svg)} chars, {n_paths} paths)")

    print(f"\nResults:")
    print(f"  Generated OK      : {stats['ok']}")
    print(f"  Failed diffusion  : {stats['fail_diffusion']}")
    print(f"  Failed vectorize  : {stats['fail_vectorize']}")
    print(f"  Failed quality    : {stats['fail_quality']}")

    if failed:
        print("\nFailed prompts:")
        for f_ in failed:
            print(f"  [{f_['reason']:10s}] {f_['prompt']}")

In [ ]:
#@title Step 9 — Preview Generated SVGs
import re
import io
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def render_svg(svg_string, size=200):
    try:
        import cairosvg
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cairosvg"])
        import cairosvg
    try:
        png = cairosvg.svg2png(bytestring=svg_string.encode(), output_width=size, output_height=size)
        return Image.open(io.BytesIO(png)).convert("RGB")
    except Exception:
        return None

show = new_pairs[:min(10, len(new_pairs))]
if not show:
    print("No successful pairs to preview.")
else:
    cols = min(5, len(show))
    rows = (len(show) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten() if rows * cols > 1 else [axes]
    for ax, pair in zip(axes, show):
        rendered = render_svg(pair["svg"])
        if rendered:
            ax.imshow(rendered)
        ax.set_title(pair["prompt"][:22], fontsize=8)
        ax.axis("off")
    for ax in axes[len(show):]:
        ax.axis("off")
    plt.suptitle("Generated corrective SVGs", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
#@title Step 10 — Merge & Download training_pairs.json
import json
from google.colab import files

# Load existing seed pairs if uploaded
existing = []
if EXISTING_PAIRS_PATH and __import__('os').path.exists(EXISTING_PAIRS_PATH):
    with open(EXISTING_PAIRS_PATH, encoding="utf-8") as f:
        existing = json.load(f)
    print(f"Existing seed pairs   : {len(existing)}")

# Merge: existing seeds first, then new corrective pairs
# Deduplicate by prompt (keep existing if duplicate)
seen_prompts = {item["prompt"] for item in existing}
unique_new   = [p for p in new_pairs if p["prompt"] not in seen_prompts]

merged = existing + unique_new

print(f"New corrective pairs  : {len(new_pairs)}")
print(f"Unique new (no dupes) : {len(unique_new)}")
print(f"Total merged          : {len(merged)}")

# Complexity breakdown
def classify_complexity(svg):
    n = len(__import__('re').findall(
        r'<(rect|circle|ellipse|polygon|polyline|line|path|text)\b', svg))
    return "simple" if n <= 3 else ("medium" if n <= 10 else "complex")

stats = {}
for item in merged:
    c = classify_complexity(item["svg"])
    item["complexity"] = c
    stats[c] = stats.get(c, 0) + 1
print(f"Complexity breakdown  : {stats}")

# Save
out_path = "/content/training_pairs.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {out_path}")

# Download
files.download(out_path)
print("\n✅ Download started!")
print("Upload this training_pairs.json in Step 3 of DiffuSVG_v7_Colab.ipynb.")